# MetaPilot Phase 2: QLoRA pipeline ranking (Colab, T4)

Runtime → Change runtime type → **T4 GPU**. Then *Run all*.

* Upload `phase2_bundle.zip` to `MyDrive/metapilot_phase2/` first.
* Everything this notebook produces is written to that Drive folder:
  `predictions/predictions_{base,A,B}_fold{0,1,2}.json` and `adapters/{A,B}_fold{f}/`.
* **Resumable:** if Colab disconnects, reconnect and *Run all* again. Each (method, fold) whose
  predictions file already exists is skipped, and so is each adapter that already finished.
* Order: base model (folds 0-2), then fine-tuned A (folds 0-2), then fine-tuned B (folds 0-2).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "trl>=0.19" "peft>=0.13" "bitsandbytes>=0.45" "accelerate>=1.0" datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, zipfile, json, time, gc
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/metapilot_phase2')
BUNDLE = DRIVE_DIR / 'phase2_bundle.zip'
WORK = Path('/content/metapilot')
PRED_DIR = DRIVE_DIR / 'predictions'
ADAPTER_DIR = DRIVE_DIR / 'adapters'
PRED_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

assert BUNDLE.exists(), f'Upload phase2_bundle.zip to {DRIVE_DIR} first'
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK))

from src.llm_infer import load_model, predict_file, BASE_MODEL
from src.llm_finetune import train_adapter, HPARAMS

P2 = WORK / 'data' / 'phase2'
SYSTEM_PROMPT = (P2 / 'system_prompt.txt').read_text()
FOLDS = [0, 1, 2]
VARIANTS = ['A', 'B']
print('base model:', BASE_MODEL); print('hparams:', HPARAMS)
print('done so far:', sorted(p.name for p in PRED_DIR.glob('predictions_*.json')))

In [ ]:
import torch

def free():
    gc.collect(); torch.cuda.empty_cache()

def pred_path(method, f):
    return PRED_DIR / f'predictions_{method}_fold{f}.json'

# 1) Base model, no fine-tuning, prompt A
todo = [f for f in FOLDS if not pred_path('base', f).exists()]
if todo:
    model, tok = load_model(adapter_dir=None)
    for f in todo:
        t0 = time.time()
        predict_file(model, tok, SYSTEM_PROMPT, P2 / 'prompts' / 'A' / f'fold{f}_test.jsonl',
                     pred_path('base', f), method='base', fold=f)
        print(f'base fold{f}: {time.time() - t0:.0f}s')
    del model, tok; free()
else:
    print('base: all folds done')

In [ ]:
# 2) Fine-tune + predict for each variant x fold
for v in VARIANTS:
    for f in FOLDS:
        out = pred_path(v, f)
        if out.exists():
            print(f'{v} fold{f}: predictions exist, skipped'); continue
        adapter = ADAPTER_DIR / f'{v}_fold{f}'
        if not (adapter / 'adapter_config.json').exists():
            t0 = time.time()
            # train file contains only tasks from folds != f (built by src/serialize_text.py)
            train_adapter(P2 / 'prompts' / v / f'fold{f}_train.jsonl', adapter, SYSTEM_PROMPT)
            print(f'{v} fold{f}: trained in {(time.time() - t0) / 60:.1f} min'); free()
        t0 = time.time()
        model, tok = load_model(adapter_dir=str(adapter))
        predict_file(model, tok, SYSTEM_PROMPT, P2 / 'prompts' / v / f'fold{f}_test.jsonl',
                     out, method=v, fold=f)
        print(f'{v} fold{f}: predicted in {time.time() - t0:.0f}s')
        del model, tok; free()

In [ ]:
# 3) Check: all 9 prediction files present -> download the predictions folder
expected = [f'predictions_{m}_fold{f}.json' for m in ['base'] + VARIANTS for f in FOLDS]
have = {p.name for p in PRED_DIR.glob('predictions_*.json')}
print('missing:', [e for e in expected if e not in have] or 'none')
sample = json.loads(pred_path('A', 0).read_text())['predictions'][:3] if pred_path('A', 0).exists() else []
for s in sample: print(s['task_id'], '->', s['generated'])